# Artículo: Limpieza de predictores socioeconómicos y demográficos
* **Nombre**: Sierra Wells  
* **Clase**: Programación (Geointeligencia Computacional)
* **Programa**: Maestría en Ciencias de Información Geoespacial
* **Fecha**: 19 noviembre 2026

## 📝 Descripción
Lee los principales resultados por localidad (ITER) del Censo de Población y Vivienda 2010 y construye, para cada **municipio**, la población total y los indicadores socioeconómicos y demográficos que se usarán como predictores en el modelo MCO (OLS) de homicidios de 2018.

Fuente: [INEGI, Censo de Población y Vivienda 2010](https://www.inegi.org.mx/programas/ccpv/2010/#datos_abiertos).

Resultado en `clean/output/`:
* `2_clean_sociodem.csv`: indicadores por municipio (`cve_mun`).

In [1]:
# Librerías
from pathlib import Path

import pandas as pd

In [2]:
# Rutas (el notebook se corre desde clean/src/)
TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / 'input'
OUTPUT_DIR = TASK_DIR / 'output'

In [3]:
# Variables del censo que se usan para construir los indicadores
ID_COLUMNS = ['entidad', 'nom_ent', 'mun', 'nom_mun', 'loc', 'nom_loc']
COUNT_COLUMNS = [
    'pobtot', 'p_3ymas', 'p_5ymas', 'p_12ymas',
    'p_15a17_m', 'p_18a24_m',
    'graproes', 'pea', 'pdesocup', 'tothog', 'hogjef_f',
    'presoe05', 'pnacoe', 'p3ym_hli', 'psinder', 'pder_imss', 'p12ym_sepa',
    'tvivparhab', 'vph_snbien', 'vph_pisoti',
]

# Localidades con al menos 2,500 habitantes (definición de localidad urbana del INEGI)
URBAN_MIN_POP = 2_500

**1. Leer datos**

El ITER tiene una fila por localidad y, además, filas con totales:
* `entidad == '00'`: total nacional.
* `mun == '000'` y `loc == '0000'`: total de cada entidad (`nom_loc` es "Total de la Entidad").
* `loc == '0000'`: total de cada municipio (`nom_loc` es "Total del Municipio").
* `loc == '9998'` y `loc == '9999'`: localidades de una y de dos viviendas.

Todas las columnas se leen como texto para no perder los ceros a la izquierda de las claves.

In [4]:
census = pd.read_csv(INPUT_DIR / 'iter_00_cpv2010.csv', usecols=ID_COLUMNS + COUNT_COLUMNS, dtype=str, encoding='utf-8-sig')
census.head()

,entidad,nom_ent,mun,nom_mun,loc,nom_loc,pobtot,p_3ymas,p_5ymas,p_12ymas,...,pea,pdesocup,psinder,pder_imss,p12ym_sepa,tothog,hogjef_f,tvivparhab,vph_pisoti,vph_snbien
0,00,Total nacional,000,Total nacional,0000,Total nacional,112336538,104781265,100410810,84927468,...,44701044,2031369,38020372,35380021,8162339,28159373,6916206,28607568,1731414,782716
1,00,Total nacional,000,Total nacional,9998,Localidades de una vivienda,260087,245319,236360,205923,...,114327,1658,120820,42422,17302,63531,5588,64241,13552,8196
2,00,Total nacional,000,Total nacional,9999,Localidades de dos viviendas,173056,162593,156057,133734,...,67165,1407,79539,23108,10477,39674,4853,39979,10604,6127
3,01,Aguascalientes,000,Total de la entidad Aguascalientes,0000,Total de la Entidad,1184996,1109480,1059407,879679,...,475207,31381,249596,530445,71817,289575,64932,290777,4802,1300
4,01,Aguascalientes,000,Total de la entidad Aguascalientes,9998,Localidades de una vivienda,3395,3191,3043,2517,...,1273,40,1031,877,157,799,83,802,51,26


**2. Quedarse con los totales por municipio**

Los indicadores se calculan con las filas de totales de cada municipio. Así, los porcentajes se calculan con los conteos de todo el municipio.

In [5]:
# Totales por municipio (clave de 5 dígitos: 2 de la entidad y 3 del municipio)
muns = census.loc[(census['loc'] == '0000') & (census['mun'] != '000')].copy()
muns['cve_ent'] = muns['entidad']
muns['cve_mun'] = muns['entidad'] + muns['mun']

# Pasar los conteos a números
for column in COUNT_COLUMNS:
    muns[column] = pd.to_numeric(muns[column])

# Revisiones: 2,456 municipios, y su población suma el total nacional
national_total = int(census.loc[census['entidad'] == '00', 'pobtot'].iloc[0])
assert len(muns) == 2456 and muns['cve_mun'].is_unique
assert muns['pobtot'].sum() == national_total

Para el porcentaje de población urbana se usan las filas de localidades (sin los totales ni los agregados `9998` y `9999`): una localidad es urbana si tiene al menos 2,500 habitantes.

In [6]:
is_locality = ~census['loc'].isin(['0000', '9998', '9999']) & (census['mun'] != '000')
localities = census.loc[is_locality, ['entidad', 'mun', 'pobtot']].copy()
localities['pobtot'] = pd.to_numeric(localities['pobtot'])
localities['cve_mun'] = localities['entidad'] + localities['mun']

# Población en localidades urbanas, por municipio
urban = localities.loc[localities['pobtot'] >= URBAN_MIN_POP]
urban_pop_mun = urban.groupby('cve_mun')['pobtot'].sum()

**3. Construir indicadores**

Los porcentajes usan como denominador la población a la que se refiere cada variable del censo:

| Variable | Definición | Variables del censo |
|---|---|---|
| `pob_tot` | Población total | `pobtot` |
| `pct_men_15_24` | % de la población que son hombres de 15 a 24 años | (`p_15a17_m` + `p_18a24_m`) / `pobtot` |
| `avg_schooling` | Grado promedio de escolaridad (población de 15 años y más) | `graproes` |
| `pct_unemployed` | % de la población económicamente activa (12 años y más) que está desocupada | `pdesocup` / `pea` |
| `pct_female_head` | % de hogares censales con jefatura femenina | `hogjef_f` / `tothog` |
| `pct_moved_since_2005` | % de la población de 5 años y más que en junio de 2005 vivía en otra entidad | `presoe05` / `p_5ymas` |
| `pct_indigenous_lang` | % de la población de 3 años y más que habla alguna lengua indígena | `p3ym_hli` / `p_3ymas` |
| `pct_no_health_insurance` | % de la población sin derechohabiencia a servicios de salud | `psinder` / `pobtot` |
| `pct_imss` | % de la población derechohabiente del IMSS (indicador de empleo formal) | `pder_imss` / `pobtot` |
| `pct_urban` | % de la población que vive en localidades de 2,500 habitantes o más | población de las localidades urbanas / `pobtot` |
| `pct_born_other_state` | % de la población nacida en otra entidad | `pnacoe` / `pobtot` |
| `pct_separated` | % de la población de 12 años y más separada, divorciada o viuda | `p12ym_sepa` / `p_12ymas` |
| `pct_no_goods` | % de viviendas particulares habitadas sin ningún bien (radio, televisión, refrigerador, etc.) | `vph_snbien` / `tvivparhab` |
| `pct_dirt_floor` | % de viviendas particulares habitadas con piso de tierra | `vph_pisoti` / `tvivparhab` |

In [7]:
def build_indicators(totals, urban_pop):
    # Indicadores de la tabla anterior, a partir de las filas de totales de los municipios
    return pd.DataFrame({
        'pob_tot': totals['pobtot'],
        'pct_men_15_24': (totals['p_15a17_m'] + totals['p_18a24_m']) / totals['pobtot'] * 100,
        'avg_schooling': totals['graproes'],
        'pct_unemployed': totals['pdesocup'] / totals['pea'] * 100,
        'pct_female_head': totals['hogjef_f'] / totals['tothog'] * 100,
        'pct_moved_since_2005': totals['presoe05'] / totals['p_5ymas'] * 100,
        'pct_indigenous_lang': totals['p3ym_hli'] / totals['p_3ymas'] * 100,
        'pct_no_health_insurance': totals['psinder'] / totals['pobtot'] * 100,
        'pct_imss': totals['pder_imss'] / totals['pobtot'] * 100,
        'pct_urban': urban_pop / totals['pobtot'] * 100,
        'pct_born_other_state': totals['pnacoe'] / totals['pobtot'] * 100,
        'pct_separated': totals['p12ym_sepa'] / totals['p_12ymas'] * 100,
        'pct_no_goods': totals['vph_snbien'] / totals['tvivparhab'] * 100,
        'pct_dirt_floor': totals['vph_pisoti'] / totals['tvivparhab'] * 100,
    })


# Los municipios sin localidades urbanas tienen 0% de población urbana)
sociodem_mun = pd.concat([
    muns[['cve_mun', 'cve_ent', 'nom_ent', 'nom_mun']],
    build_indicators(muns, muns['cve_mun'].map(urban_pop_mun).fillna(0).values),
], axis=1).reset_index(drop=True)

# Revisiones: sin valores faltantes y porcentajes entre 0 y 100
pct_columns = [column for column in sociodem_mun.columns if column.startswith('pct_')]
assert sociodem_mun.notna().all().all()
assert sociodem_mun[pct_columns].ge(0).all().all()
assert sociodem_mun[pct_columns].le(100).all().all()

sociodem_mun.head()

,cve_mun,cve_ent,nom_ent,nom_mun,pob_tot,pct_men_15_24,avg_schooling,pct_unemployed,pct_female_head,pct_moved_since_2005,pct_indigenous_lang,pct_no_health_insurance,pct_imss,pct_urban,pct_born_other_state,pct_separated,pct_no_goods,pct_dirt_floor
0,01001,01,Aguascalientes,Aguascalientes,797010,9.407661,9.81,6.237870,24.029820,4.568117,0.239020,22.392567,52.184665,92.932335,23.459179,8.976156,0.247452,1.284773
1,01002,01,Aguascalientes,Asientos,45492,9.909435,7.25,7.591312,16.208654,1.685170,0.111235,13.769454,19.763915,28.033500,11.417392,6.181931,1.788033,2.796923
2,01003,01,Aguascalientes,Calvillo,54136,9.169499,6.90,9.704816,22.884525,1.711101,0.193489,17.110610,11.332570,49.238954,11.783287,6.621886,0.599502,2.545936
3,01004,01,Aguascalientes,Cosío,15042,9.646324,7.83,8.528740,15.670996,2.122888,0.120893,13.907725,18.541417,32.562159,9.161016,5.656884,1.153735,3.461206
4,01005,01,Aguascalientes,Jesús María,99590,9.248921,8.60,4.924329,17.491011,4.048416,0.194181,22.332564,46.479566,68.355257,16.367105,6.216950,0.514902,2.469782


**4. Guardar resultados**

In [8]:
sociodem_mun.to_csv(OUTPUT_DIR / '2_clean_sociodem.csv', index=False)